<a href="https://colab.research.google.com/github/UnfoldDataScience/Agentic_Ai_For_Beginner/blob/main/Part2/First_Multi_Agent_Part_2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: August 2026 (Tested on Google Colab)

# ⚠️ Important: If running in Colab , Switch to Python 3.12

Before running any other cells in this notebook, please switch your Google Colab runtime to **Python 3.12**.

From the top right arrow (next to Connect/RAM), select:

**Change runtime type → Runtime Version → 2026.07**

The next cell should output python version as 3.12


In [1]:
!python --version

Python 3.12.13


# IMPORTANT

1. Run the installation cell first
2. Add required API keys in Colab Secrets
3. Run notebook cells sequentially

# If running in Google Colab Please ensure you update below keys in "secrets" on the left and give access to this notebook

1.   OPENAI_API_KEY

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

> 💡 **Note:** While running package installation command below,Google Colab may display some dependency messages during installation. These relate to packages we are not using in this notebook, so you can safely continue when the installation completes.

In [3]:
# Install Part 2 — CrewAI dependencies

!pip install -q \
    "crewai==0.28.8" \
    "crewai_tools==0.1.6" \
    "langchain_community==0.0.29" \
    "openai==1.59.6" \
    "python-dotenv==1.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.7/160.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.5/525.5 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 16.7 MB/s 

In [ ]:
#below lines are needed for restarting kernel in Colab -
# If you are in local environment you can use "restart" button on top of the notebook
import os
os.kill(os.getpid(), 9)

In [1]:
import openai
import crewai
from crewai import Agent, Task, Crew

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:949: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warnings.warn(


In [2]:
#If executing from local machine, run below 2 lines to load keys (.env should be present in same directory with keys in it)
# from dotenv import load_dotenv
# load_dotenv()


#If executing from Colab, run below lines to load keys (keys should be added in colab secrets and access given to this notebook)
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [3]:
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'

In [4]:
#1- Chef
chef = Agent(
    role="chef",
    goal="Give 5 different receipes using ingredients : {ingredients}",
    backstory="You're  a Chef who has experience on cooking delicious foods"
              "Your expertice lies in recommeding some new foods to the customers"
              "Please generate and give receipe of 5 different foods using this ingredients : {ingredients} "
              "describing stepwise process, time to cook, other ingredients needed and calorie count"
              "Your recipe will be critically analyzed by nutritutionist to recommend food for diabetes patients ",
	verbose=True
)

In [5]:
2# Nutritionist
nutritionist = Agent(
    role="nutritionist",
    goal="Critically analyze the recipe given by chef using these ingredients: {ingredients}",
    backstory="You're a nutritionist and have good knowledge on working with Diabetic patients "
              "There are recipe recommended by chef using these ingredients : {ingredients}. "
              "please critically analyze the recipe and suggest best food out of goven 5 for Diabetic patients"
              "Also give reasons why you dont recommend other 4 receips and choose this one",
    verbose=True
)

In [6]:
cook = Task(
    description=(
        "1. Write receipe of 5 different foods using these ingredients  : {ingredients}.\n"
    ),
    expected_output="5 different receipes using {ingredients}"
                    "Stepwise guide to cook"
                    "List of all additional ingredients required for cooking"
                    "The expected taste of the food",
    agent=chef,
)

In [7]:
recommend = Task(
    description=(
        "1. Read the receipes provided by chef using {ingredients} \n"
        "2. critically analyze all receipes keeping in mind what should suit to diabetic patients.\n"
        "3. Suggest one best food out of all 5 receipes given by chef.\n"
        "4. Prove that other 4 receipes suggested by chef is not good for diabetic patients.\n"

    ),
    expected_output="A well-written critical pointwise analysis of all 5 recipes using {ingredients}"
                    "Suggest one receipe to diabetic patients based on your own knowledge and evidence "
                    "Reject all other 4 receipes with knowledge and evidence "
                    "Finally, give the stepwoise receipe of what you selected and a nice message",
    agent=nutritionist,
)

In [8]:
crewaman = Crew(
    agents=[chef,nutritionist],
    tasks=[cook, recommend],
    verbose=2
)

In [10]:
result = crewaman.kickoff(inputs={"ingredients": "chicken"})

 [DEBUG]: == Working Agent: chef
 [INFO]: == Starting Task: 1. Write receipe of 5 different foods using these ingredients  : chicken.



> Entering new CrewAgentExecutor chain...
I need to create five different recipes using chicken, ensuring that they are suitable for diabetic patients. I will outline the stepwise process for cooking each dish, list additional ingredients required, and describe the expected taste. After compiling the recipes, I will consult with a nutritionist regarding their suitability for diabetes management.

### Recipe 1: Grilled Lemon Herb Chicken

#### Ingredients:
- 500g chicken breast
- 2 tablespoons olive oil
- Juice of 1 lemon
- 2 cloves garlic, minced
- 1 teaspoon dried oregano
- Salt and pepper to taste
- Fresh parsley for garnish

#### Cooking Steps:
1. In a bowl, mix olive oil, lemon juice, minced garlic, oregano, salt, and pepper.
2. Add chicken breasts to the marinade, ensuring they are well coated. Let it marinate for at least 30 minutes.
3. Preheat 